# Italy Dataset Builder

Reads the Italy CSV produced by the scraper, selectively extracts only the Italian
images from the downloaded shard TAR archives, and writes three compact re-packed TAR
files (one per macro-region) plus a master CSV. The region TARs are what the training
notebook loads at the start of each Colab session.

## Pipeline overview

```
Scraper CSVs  +  Shard TAR archives (GLDv2)
        │
        ▼
  [2] Load & validate CSV          ← bounding-box pre-filtered by scraper
  [2b] Italy polygon filter        ← point-in-polygon against actual land border
  [3] Discover available shards    ← match shard archives on Drive
  [4] Selective extraction         ← pull only Italian images from each shard TAR
  [5] Enrich metadata              ← join attribution / hierarchy CSVs (optional)
  [6] Write master CSV             ← italy_master.csv with lat/lon/region/shard/…
  [7] Re-pack into region TARs     ← italy_nord.tar / italy_centro.tar / italy_sud_isole.tar
```

## Section index

| # | Section | When to run |
|---|---------|-------------|
| 1 | Setup & configuration | Every session |
| 2 | Load & validate Italy CSV | Every session |
| 2b | Italy polygon filter | Every session (runs automatically after Section 2) |
| 3 | Discover available shard archives | Every session |
| 4 | Selective extraction | Every session (already-extracted images are skipped) |
| 5 | Enrich with metadata | Optional — skip if metadata files are not on Drive |
| 6 | Write master CSV | After extraction (or after re-running the scraper) |
| 7 | Re-pack into region TARs | After a new extraction or when adding more shards |
| 8 | Summary & verification | After repacking |

**Resumable**: already-extracted images are skipped in Section 4. Re-run any time new
shard archives arrive on Drive.

## Section 1 — Setup & Configuration

In [ ]:
!pip install -q pandas tqdm geopandas
print('Packages ready.')

In [ ]:
import os

# ============================================================
# GLOBAL CONFIGURATION — edit only this cell
# ============================================================

DRIVE_BASE = '/content/drive/MyDrive/Vision_Project_2026'

# Input: Italy CSV produced by the scraper
# Accepts either the italy-only file or the full dataset (will filter by is_italy)
ITALY_CSV_PATH   = f'{DRIVE_BASE}/GLDv2/gld_italy_dataset.csv'
FULL_CSV_PATH    = f'{DRIVE_BASE}/GLDv2/gld_full_dataset.csv'   # fallback if italy csv missing

# Input: shard TAR archives produced by the scraper
ARCHIVE_DIR      = f'{DRIVE_BASE}/GLDv2/archives'

# Output: where to write the organised Italy images on Drive
ITALY_DIR        = f'{DRIVE_BASE}/ItalyDataset'
IMAGES_DIR       = f'{ITALY_DIR}/images'          # individual JPGs, organised by region/
TARS_DIR         = f'{ITALY_DIR}/archives'         # three region TAR files

# Output CSV
MASTER_CSV       = f'{ITALY_DIR}/italy_master.csv'

# Optional: metadata files (downloaded by GLDv2_Metadata_Analysis.ipynb)
# Set to None to skip enrichment if they are not available
ATTRIBUTION_CSV  = f'{DRIVE_BASE}/GLDv2/metadata/train_attribution.csv'    # or None
HIERARCHICAL_CSV = f'{DRIVE_BASE}/GLDv2/metadata/train_label_to_hierarchical.csv'  # or None
TRAIN_CSV        = f'{DRIVE_BASE}/GLDv2/metadata/train.csv'                # needed for landmark_id lookup

# Italy bounding box (must match the scraper that produced the CSV)
MIN_LAT, MAX_LAT = 35.4,  47.2
MIN_LON, MAX_LON =  6.6,  18.8

# Region boundaries (matches GeoClip_Training / GeoClip_Evaluation)
REGION_NORD_LAT   = 44.0
REGION_CENTRO_LAT = 41.5

# Macro-region labels used throughout the notebook (must stay consistent)
REGIONS = ['nord', 'centro', 'sud_isole']

# If True, re-pack region TARs even if they already exist
FORCE_REPACK = True

# -- Polygon filter --
# Uses Italy's actual land polygon (MultiPolygon: mainland + Sicily + Sardinia
# + all islands) to remove coordinates that are inside the bounding box but
# belong to another country: France, Monaco, Switzerland, Austria, Slovenia,
# Croatia, Bosnia, Albania, Tunisia, Libya, Algeria, Malta, Greece, etc.
# POLY_BUFFER_DEG adds a small tolerance around the border so coastal and
# border-region landmarks are not accidentally clipped. 0.02° ≈ 2 km.
ENFORCE_POLYGON_FILTER = True
POLY_BUFFER_DEG        = 0.02   # degrees; set 0 for a strict border

# Shard filter for repacking.
# Set to an integer N to repack only images whose shard index is <= N.
# Set to None to repack everything in master_df (default).
# Example: REPACK_SHARD_MAX = 100  ->  includes shards 0-100 only.
REPACK_SHARD_MAX = 100

# Create output directories
for d in [ITALY_DIR, IMAGES_DIR, TARS_DIR,
          f'{IMAGES_DIR}/nord', f'{IMAGES_DIR}/centro', f'{IMAGES_DIR}/sud_isole']:
    os.makedirs(d, exist_ok=True)

print('Configuration loaded.')
print(f'  Italy images dir : {IMAGES_DIR}')
print(f'  Region TARs dir  : {TARS_DIR}')
print(f'  Master CSV       : {MASTER_CSV}')

In [ ]:
from google.colab import drive
import os, shutil

if os.path.exists('/content/drive') and os.path.ismount('/content/drive'):
    drive.flush_and_unmount()

if os.path.exists('/content/drive') and os.listdir('/content/drive'):
    for item in os.listdir('/content/drive'):
        p = os.path.join('/content/drive', item)
        shutil.rmtree(p) if os.path.isdir(p) else os.unlink(p)

drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

## Section 2 — Load & Validate Italy CSV

Loads the Italy CSV produced by the scraper. The CSV was already coarsely filtered by
the scraper to the Italy bounding box; this section normalises column names, drops rows
with missing coordinates, and prints a basic sanity check on the coordinate ranges.

Two CSVs are supported:
- **`gld_italy_dataset.csv`** — italy-only rows already filtered by the scraper (preferred)
- **`gld_full_dataset.csv`** — full dataset; filtered here by `is_italy` flag or bounding box

In [ ]:
import pandas as pd

# Prefer the dedicated Italy CSV; fall back to full dataset with filtering
if os.path.exists(ITALY_CSV_PATH):
    print(f'Loading Italy CSV: {ITALY_CSV_PATH}')
    raw = pd.read_csv(ITALY_CSV_PATH)
    print(f'  Rows: {len(raw):,}')
elif os.path.exists(FULL_CSV_PATH):
    print(f'Italy CSV not found. Loading full dataset and filtering: {FULL_CSV_PATH}')
    raw = pd.read_csv(FULL_CSV_PATH)
    before = len(raw)
    if 'is_italy' in raw.columns:
        raw = raw[raw['is_italy'] == True]
    else:
        raw = raw[
            raw['lat'].notna() & raw['lon'].notna() &
            (raw['lat'] >= MIN_LAT) & (raw['lat'] <= MAX_LAT) &
            (raw['lon'] >= MIN_LON) & (raw['lon'] <= MAX_LON)
        ]
    print(f'  Rows after Italy filter: {before:,} -> {len(raw):,}')
else:
    raise FileNotFoundError(
        f'Neither {ITALY_CSV_PATH} nor {FULL_CSV_PATH} found.\n'
        'Run the scraper notebook first.'
    )

# Normalise column names
rename = {c: c.lower() for c in raw.columns}
rename.update({c: 'lat' for c in raw.columns if c.lower() == 'latitude'})
rename.update({c: 'lon' for c in raw.columns if c.lower() == 'longitude'})
raw = raw.rename(columns=rename)

required = {'image_id', 'lat', 'lon', 'shard'}
missing  = required - set(raw.columns)
if missing:
    raise ValueError(f'CSV is missing required columns: {missing}')

# Keep only successfully scraped images
if 'status' in raw.columns:
    raw = raw[raw['status'] == 'success']

italy_df = raw.dropna(subset=['image_id', 'lat', 'lon', 'shard']).copy()
italy_df['image_id'] = italy_df['image_id'].astype(str)
italy_df['shard']    = italy_df['shard'].astype(int)

print(f'\nClean Italy rows: {len(italy_df):,}')
print(f'Unique shards   : {italy_df["shard"].nunique()}')
print(f'Lat range       : [{italy_df["lat"].min():.3f}, {italy_df["lat"].max():.3f}]')
print(f'Lon range       : [{italy_df["lon"].min():.3f}, {italy_df["lon"].max():.3f}]')

In [ ]:
# Section 2b — Italy polygon filter
#
# The bounding box [lat 35.4–47.2, lon 6.6–18.8] also encloses parts of:
#   France / Monaco, Switzerland, Austria, Slovenia, Croatia, Bosnia, Albania,
#   Tunisia, Libya, Algeria, Malta, and Greece (Corfu).
# This cell keeps only coordinates that fall within Italy's actual land polygon
# (a MultiPolygon covering mainland + Sicily + Sardinia + all smaller islands).
#
# The optional POLY_BUFFER_DEG tolerance prevents clipping coastal landmarks
# whose Wikimedia GPS sits a few hundred metres offshore due to rounding.

if not ENFORCE_POLYGON_FILTER:
    print("Polygon filter skipped (ENFORCE_POLYGON_FILTER = False).")
else:
    import geopandas as gpd
    import warnings

    italy_geom = None

    # Method 1: geopandas < 1.0 — bundled naturalearth_lowres
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            _world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
        italy_geom = _world[_world['name'] == 'Italy'].geometry.iloc[0]
        print("Loading Italy land polygon … (naturalearth_lowres, bundled)")
    except Exception:
        pass

    # Method 2: geopandas >= 1.0 — download ne_110m_admin_0_countries from Natural Earth S3
    # This is the same source file that naturalearth_lowres was originally derived from.
    if italy_geom is None:
        print("Loading Italy land polygon … (downloading ne_110m_admin_0_countries from Natural Earth)")
        _url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
        try:
            _world = gpd.read_file(_url)
        except Exception as _e:
            raise RuntimeError(
                f"Could not download the Natural Earth countries file: {_e}\n"
                "Fix: set ENFORCE_POLYGON_FILTER = False in the config cell to skip this step."
            ) from _e
        # Downloaded shapefile uses 'NAME' (uppercase); handle both to be safe
        _name_col = next(
            (c for c in _world.columns if c.upper() == 'NAME' and 'LONG' not in c.upper()),
            'NAME'
        )
        italy_geom = _world[_world[_name_col] == 'Italy'].geometry.iloc[0]
        print("  Downloaded successfully.")

    # Optional buffer to tolerate GPS imprecision near the coast / borders
    if POLY_BUFFER_DEG > 0:
        italy_geom = italy_geom.buffer(POLY_BUFFER_DEG)
        print(f"  Coast/border buffer: {POLY_BUFFER_DEG}° ≈ {POLY_BUFFER_DEG * 111:.0f} km")

    # Vectorised point-in-polygon (STRtree under the hood — much faster than apply)
    pts_gdf = gpd.GeoDataFrame(
        italy_df,
        geometry=gpd.points_from_xy(italy_df['lon'], italy_df['lat']),
        crs='EPSG:4326',
    )
    mask = pts_gdf.geometry.within(italy_geom)

    before     = len(italy_df)
    removed_df = italy_df[~mask.values].copy()
    italy_df   = italy_df[mask.values].reset_index(drop=True)
    removed    = before - len(italy_df)

    print(f"\nPolygon filter result:")
    print(f"  Before  : {before:,}")
    print(f"  After   : {len(italy_df):,}")
    print(f"  Removed : {removed:,}  ({removed / before * 100:.1f}%)")

    if removed > 0:
        print(f"\nRemoved coordinates (first 30):")
        sample = removed_df[['image_id', 'lat', 'lon', 'shard']].head(30)
        print(sample.to_string(index=False))
        if removed > 30:
            print(f"  … and {removed - 30} more")

    del pts_gdf  # free memory

In [ ]:
def assign_region(lat):
    if lat > REGION_NORD_LAT:
        return 'nord'
    elif lat > REGION_CENTRO_LAT:
        return 'centro'
    else:
        return 'sud_isole'

italy_df['region'] = italy_df['lat'].apply(assign_region)

print('Region breakdown:')
print(italy_df['region'].value_counts().to_string())

## Section 3 — Discover Available Shard Archives

Checks which GLDv2 shard TAR archives are already downloaded on Drive. Only shards
that are present will contribute images. If some shards are missing, the notebook
continues with whatever is available and reports the gap — run the scraper to fill it.

In [ ]:
from pathlib import Path

def find_archive(shard_idx, archive_dir):
    """Return path to the TAR archive for a shard, or None."""
    for fmt in [
        f'train_images_{shard_idx:03d}.tar',
        f'images_{shard_idx:03d}.tar',
        f'train_images_{shard_idx}.tar',
    ]:
        p = os.path.join(archive_dir, fmt)
        if os.path.exists(p):
            return p
    return None

# Shards the Italy CSV references
needed_shards  = sorted(italy_df['shard'].unique())

available, unavailable = [], []
for s in needed_shards:
    path = find_archive(s, ARCHIVE_DIR)
    if path:
        size_mb = os.path.getsize(path) / 1e6
        available.append((s, path, size_mb))
    else:
        unavailable.append(s)

total_archive_gb = sum(x[2] for x in available) / 1024
print(f'Shards referenced in Italy CSV : {len(needed_shards)}')
print(f'Archives found                 : {len(available)}')
print(f'Archives missing (not yet DL)  : {len(unavailable)}')
print(f'Total archive size             : {total_archive_gb:.2f} GB')

# Images covered by available archives
available_shard_ids = {s for s, _, _ in available}
covered_df = italy_df[italy_df['shard'].isin(available_shard_ids)]
print(f'\nItaly images in available shards: {len(covered_df):,} / {len(italy_df):,}')

if unavailable:
    print(f'\nMissing shards (first 20): {unavailable[:20]}')
    print('Run the scraper to download them, then re-run this notebook.')

## Section 4 — Selective Extraction (Italy Images Only)

For each shard archive, extract **only** the Italy image files by matching member names
against the `image_id` set. Already-extracted images are skipped.

In [ ]:
import tarfile
from tqdm import tqdm


def region_image_dir(region):
    return os.path.join(IMAGES_DIR, region)


def dest_path(image_id, region):
    return os.path.join(region_image_dir(region), f'{image_id}.jpg')


def extract_italy_from_shard(tar_path, id_to_region):
    """
    Open a shard TAR and copy only the Italy images to their region directories.
    Returns (extracted_count, skipped_count).
    """
    extracted = 0
    skipped   = 0

    with tarfile.open(tar_path, 'r') as tar:
        for member in tar.getmembers():
            if not member.isfile():
                continue
            name = member.name
            if not name.lower().endswith('.jpg'):
                continue

            img_id = Path(name).stem
            if img_id not in id_to_region:
                continue

            region   = id_to_region[img_id]
            out_path = dest_path(img_id, region)

            if os.path.exists(out_path):
                skipped += 1
                continue

            # Read from TAR and write to destination
            with tar.extractfile(member) as src, open(out_path, 'wb') as dst:
                dst.write(src.read())

            extracted += 1

    return extracted, skipped


# Build image_id -> region lookup for the covered images
id_to_region = dict(zip(covered_df['image_id'], covered_df['region']))

total_extracted = 0
total_skipped   = 0
failed_shards   = []

for shard_idx, tar_path, size_mb in tqdm(available, desc='Shard archives'):
    shard_ids = set(italy_df[italy_df['shard'] == shard_idx]['image_id'])
    shard_id_to_region = {k: v for k, v in id_to_region.items() if k in shard_ids}

    if not shard_id_to_region:
        continue

    try:
        ext, skp = extract_italy_from_shard(tar_path, shard_id_to_region)
        total_extracted += ext
        total_skipped   += skp
    except Exception as e:
        print(f'  ERROR shard {shard_idx}: {e}')
        failed_shards.append(shard_idx)

print(f'\nExtraction complete.')
print(f'  Newly extracted : {total_extracted:,}')
print(f'  Already present : {total_skipped:,}')
if failed_shards:
    print(f'  Failed shards   : {failed_shards}')

In [ ]:
# Verify which images are now on disk and annotate the dataframe
print('Verifying images on disk ...')

covered_df = covered_df.copy()
covered_df['local_path'] = covered_df.apply(
    lambda r: dest_path(r['image_id'], r['region']), axis=1
)
covered_df['on_disk'] = covered_df['local_path'].apply(os.path.exists)

found   = covered_df['on_disk'].sum()
missing = len(covered_df) - found

print(f'  Images on disk  : {found:,}')
print(f'  Images missing  : {missing:,}')

print('\nBy region:')
print(covered_df.groupby('region')['on_disk'].agg(['sum', 'count']).to_string())

# Work only with images confirmed on disk
final_df = covered_df[covered_df['on_disk']].copy()
print(f'\nImages ready for CSV & packaging: {len(final_df):,}')

## Section 5 — Enrich with Attribution & Hierarchy Metadata

Optional. Adds `license`, `author`, `supercategory`, `natural_or_human_made` columns from the
GLDv2 metadata files. Skip this section if the metadata files are not available.

In [ ]:
enriched_df = final_df.copy()

# --- landmark_id lookup from train.csv ---
if TRAIN_CSV and os.path.exists(TRAIN_CSV):
    print('Loading train.csv for landmark_id lookup ...')
    train_meta = pd.read_csv(TRAIN_CSV, usecols=['id', 'landmark_id'])
    train_meta['id'] = train_meta['id'].astype(str)
    enriched_df = enriched_df.merge(
        train_meta.rename(columns={'id': 'image_id'}),
        on='image_id', how='left'
    )
    matched = enriched_df['landmark_id'].notna().sum()
    print(f'  landmark_id matched: {matched:,} / {len(enriched_df):,}')
else:
    print('train.csv not available — skipping landmark_id lookup.')
    enriched_df['landmark_id'] = None

# --- Attribution (license, author) ---
if ATTRIBUTION_CSV and os.path.exists(ATTRIBUTION_CSV):
    print('Loading train_attribution.csv ...')
    attrib = pd.read_csv(ATTRIBUTION_CSV, usecols=['id', 'license', 'author'])
    attrib['id'] = attrib['id'].astype(str)
    enriched_df = enriched_df.merge(
        attrib.rename(columns={'id': 'image_id'}),
        on='image_id', how='left'
    )
    print(f'  license matched: {enriched_df["license"].notna().sum():,}')
else:
    print('train_attribution.csv not available — skipping license/author.')
    enriched_df['license'] = None
    enriched_df['author']  = None

# --- Hierarchical (supercategory, natural_or_human_made) ---
if HIERARCHICAL_CSV and os.path.exists(HIERARCHICAL_CSV) and 'landmark_id' in enriched_df.columns:
    print('Loading train_label_to_hierarchical.csv ...')
    hier = pd.read_csv(
        HIERARCHICAL_CSV,
        usecols=['landmark_id', 'supercategory', 'natural_or_human_made']
    )
    enriched_df = enriched_df.merge(hier, on='landmark_id', how='left')
    print(f'  supercategory matched: {enriched_df["supercategory"].notna().sum():,}')
else:
    print('train_label_to_hierarchical.csv not available — skipping supercategory.')
    enriched_df['supercategory']         = None
    enriched_df['natural_or_human_made'] = None

print(f'\nEnriched dataframe: {len(enriched_df):,} rows x {enriched_df.shape[1]} columns')

## Section 6 — Write Master CSV

Assembles the final `italy_master.csv` from the extracted and enriched dataframe.
This is the single source of truth consumed by Section 7 (repacking) and by the
training notebook. Re-run this section any time the enrichment data changes.

In [ ]:
# Select and order columns for the master CSV
keep_cols = [
    'image_id',
    'local_path',       # Drive path to the extracted image
    'lat',
    'lon',
    'region',           # nord / centro / sud_isole
    'shard',
    'wiki_url',
]
optional_cols = [
    'landmark_id', 'license', 'author', 'supercategory', 'natural_or_human_made',
    'dataset',
]

final_cols = keep_cols + [c for c in optional_cols if c in enriched_df.columns]
master_df  = enriched_df[final_cols].reset_index(drop=True)

master_df.to_csv(MASTER_CSV, index=False)

print(f'Master CSV written: {MASTER_CSV}')
print(f'  Rows    : {len(master_df):,}')
print(f'  Columns : {list(master_df.columns)}')
print(f'\nFirst 3 rows:')
print(master_df.head(3).to_string())

## Section 7 — Re-pack into Region TAR Archives

Creates three TAR files:
- `italy_nord.tar`
- `italy_centro.tar`
- `italy_sud_isole.tar`

Each archive stores images flat (`{image_id}.jpg`) with no subdirectory nesting,
making them easy to load in training notebooks.

Set `FORCE_REPACK = True` in the config cell to rebuild archives that already exist.

In [ ]:
import tarfile
from tqdm import tqdm

# Apply optional shard filter before packing
if REPACK_SHARD_MAX is not None:
    repack_df = master_df[master_df['shard'] <= REPACK_SHARD_MAX].copy()
    print(f'Shard filter: shards 0-{REPACK_SHARD_MAX} only  '
          f'({len(repack_df):,} / {len(master_df):,} images)')
else:
    repack_df = master_df
    print(f'No shard filter — repacking all {len(repack_df):,} images.')

for region in REGIONS:
    tar_path = os.path.join(TARS_DIR, f'italy_{region}.tar')

    if os.path.exists(tar_path) and not FORCE_REPACK:
        size_mb = os.path.getsize(tar_path) / 1e6
        n = len(repack_df[repack_df['region'] == region])
        print(f'[{region}] Already exists ({size_mb:.0f} MB, {n:,} images from selected shards) — set FORCE_REPACK=True to rebuild.')
        continue

    region_rows = repack_df[repack_df['region'] == region]
    print(f'[{region}] Packing {len(region_rows):,} images -> {tar_path}')

    if region_rows.empty:
        print(f'  No images for region {region}, skipping.')
        continue

    packed  = 0
    missing = 0

    with tarfile.open(tar_path, 'w') as tar:
        for _, row in tqdm(region_rows.iterrows(), total=len(region_rows),
                           desc=f'  Packing {region}', leave=False):
            img_path = row['local_path']
            if not os.path.exists(img_path):
                missing += 1
                continue
            # Store flat: {image_id}.jpg, no sub-directories
            tar.add(img_path, arcname=f'{row["image_id"]}.jpg')
            packed += 1

    size_mb = os.path.getsize(tar_path) / 1e6
    print(f'  Done: {packed:,} packed, {missing:,} missing  ({size_mb:.0f} MB)')


## Section 8 — Summary & Verification

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print('=' * 60)
print('ITALY DATASET BUILDER — FINAL SUMMARY')
print('=' * 60)

print(f'\n--- Image counts by region ---')
for region in REGIONS:
    n     = (master_df['region'] == region).sum()
    pct   = n / len(master_df) * 100
    print(f'  {region:<12}: {n:>6,}  ({pct:.1f}%)')
print(f'  {"TOTAL":<12}: {len(master_df):>6,}')

print(f'\n--- TAR archive sizes ---')
for region in REGIONS:
    p = os.path.join(TARS_DIR, f'italy_{region}.tar')
    if os.path.exists(p):
        print(f'  italy_{region}.tar : {os.path.getsize(p)/1e6:,.0f} MB')
    else:
        print(f'  italy_{region}.tar : not yet created')

print(f'\n--- Output locations ---')
print(f'  Images dir  : {IMAGES_DIR}')
print(f'  Archives dir: {TARS_DIR}')
print(f'  Master CSV  : {MASTER_CSV}')

print(f'\n--- Column summary of master CSV ---')
print(master_df.dtypes.to_string())

print(f'\n--- Coordinate ranges ---')
print(f'  Lat: [{master_df["lat"].min():.4f}, {master_df["lat"].max():.4f}]')
print(f'  Lon: [{master_df["lon"].min():.4f}, {master_df["lon"].max():.4f}]')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Italy Dataset — Spatial Distribution', fontsize=13, fontweight='bold')

colors = {'nord': 'steelblue', 'centro': 'seagreen', 'sud_isole': 'tomato'}

# Scatter: lat vs lon, coloured by region
for region, grp in master_df.groupby('region'):
    axes[0].scatter(grp['lon'], grp['lat'], s=2, alpha=0.4,
                    c=colors[region], label=region)
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('GPS scatter (Italy)')
axes[0].legend(markerscale=5)
axes[0].axhline(REGION_NORD_LAT,   color='gray', lw=0.8, linestyle='--')
axes[0].axhline(REGION_CENTRO_LAT, color='gray', lw=0.8, linestyle='--')

# Bar: images per region
region_counts = master_df['region'].value_counts()[REGIONS]
bars = axes[1].bar(REGIONS, region_counts.values,
                   color=[colors[r] for r in REGIONS], edgecolor='white')
for bar, v in zip(bars, region_counts.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
                 f'{v:,}', ha='center', fontsize=10)
axes[1].set_ylabel('Images')
axes[1].set_title('Images per Region')

# Histogram: images per shard
shard_counts = master_df['shard'].value_counts().sort_index()
axes[2].bar(shard_counts.index, shard_counts.values, width=1, color='mediumpurple', edgecolor='none')
axes[2].set_xlabel('Shard index')
axes[2].set_ylabel('Italy images')
axes[2].set_title('Italy Images per Shard')

plt.tight_layout()
out = os.path.join(ITALY_DIR, 'italy_dataset_overview.png')
plt.savefig(out, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# Quick integrity check: spot-check each TAR by listing its first 5 members
print('TAR integrity spot-check:')
for region in REGIONS:
    p = os.path.join(TARS_DIR, f'italy_{region}.tar')
    if not os.path.exists(p):
        print(f'  [{region}] NOT FOUND')
        continue
    with tarfile.open(p, 'r') as tar:
        members = [m for m in tar.getmembers() if m.isfile()]
    sample = [m.name for m in members[:5]]
    print(f'  [{region}] {len(members):,} files  — sample: {sample}')

## Loading the Dataset in Training Notebooks

```python
import pandas as pd, os, tarfile

ITALY_DIR  = '/content/drive/MyDrive/Vision_Project_2026/ItalyDataset'
MASTER_CSV = f'{ITALY_DIR}/italy_master.csv'
TARS_DIR   = f'{ITALY_DIR}/archives'
LOCAL_DIR  = '/content/italy_images'

df = pd.read_csv(MASTER_CSV)  # image_id, local_path, lat, lon, region, …

# Extract a specific region archive to local disk
region = 'nord'
with tarfile.open(f'{TARS_DIR}/italy_{region}.tar', 'r') as tar:
    tar.extractall(LOCAL_DIR)

# Map image_id -> local extracted path
image_index = {
    row.image_id: os.path.join(LOCAL_DIR, f'{row.image_id}.jpg')
    for row in df[df.region == region].itertuples()
    if os.path.exists(os.path.join(LOCAL_DIR, f'{row.image_id}.jpg'))
}
print(f'{len(image_index):,} images indexed')
```

### Output structure

```
/content/drive/MyDrive/Vision_Project_2026/ItalyDataset/
├── italy_master.csv              ← master metadata table
├── italy_dataset_overview.png    ← spatial distribution charts
├── images/
│   ├── nord/       *.jpg
│   ├── centro/     *.jpg
│   └── sud_isole/  *.jpg
└── archives/
    ├── italy_nord.tar
    ├── italy_centro.tar
    └── italy_sud_isole.tar
```

### Master CSV columns

| Column | Description |
|--------|------------|
| `image_id` | 16-char hex image identifier |
| `local_path` | Drive path to the extracted JPEG |
| `lat` / `lon` | GPS coordinates |
| `region` | `nord` / `centro` / `sud_isole` |
| `shard` | Source shard index (0–499) |
| `wiki_url` | Wikimedia category URL used to scrape coordinates |
| `landmark_id` | GLDv2 landmark ID *(if train.csv available)* |
| `license` | Wikimedia license *(if train_attribution.csv available)* |
| `author` | Image author *(if train_attribution.csv available)* |
| `supercategory` | Landmark type e.g. `church building` *(if hierarchy CSV available)* |
| `natural_or_human_made` | `natural` or `human-made` *(if hierarchy CSV available)* |